# K-Fold-By-Client Strategy Comparison

Use this notebook to compare the latest available k-fold-by-client run from each preprocessing strategy and rank the combined candidates from best to worst.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd()
if (cwd / 'zscore').exists() and (cwd / 'client_zscore').exists():
    experiment_root = cwd
elif (cwd / 'experiments' / 'kfoldbyclient').exists():
    experiment_root = cwd / 'experiments' / 'kfoldbyclient'
else:
    raise FileNotFoundError('Could not locate the experiments/kfoldbyclient folder from the current working directory.')

strategies = ['zscore', 'client_zscore', 'magnitude_features', 'magnitude_only', 'robust_clip']
ranking_columns = ['pr_auc', 'miss_rate', 'far', 'balanced_accuracy', 'f1']
ranking_ascending = [False, True, True, False, False]

strategy_runs = []
for strategy in strategies:
    results_root = experiment_root / strategy / 'results'
    run_dirs = sorted([path for path in results_root.glob('run_*') if path.is_dir()]) if results_root.exists() else []
    latest_run = run_dirs[-1] if run_dirs else None
    strategy_runs.append({
        'strategy': strategy,
        'latest_run': None if latest_run is None else latest_run.name,
        'run_path': None if latest_run is None else str(latest_run),
        'available': latest_run is not None,
    })

print(f'Using experiment root: {experiment_root}')
pd.DataFrame(strategy_runs)

## Load Latest Results


In [ ]:
global_frames = []
dataset_frames = []
fold_frames = []

for row in strategy_runs:
    if not row['available']:
        continue
    run_path = Path(row['run_path'])
    metrics_global = pd.read_csv(run_path / 'metrics_global.csv')
    metrics_by_dataset = pd.read_csv(run_path / 'metrics_by_dataset.csv')
    fold_path = run_path / 'fold_model_metrics.csv'
    fold_df = pd.read_csv(fold_path) if fold_path.exists() else pd.DataFrame()

    metrics_global['preprocessing_strategy'] = row['strategy']
    metrics_by_dataset['preprocessing_strategy'] = row['strategy']
    metrics_global['model_candidate'] = metrics_global['selected_candidate']
    metrics_by_dataset['model_candidate'] = metrics_by_dataset['selected_candidate']
    metrics_global['selected_candidate_label'] = metrics_global['preprocessing_strategy'] + ' + ' + metrics_global['model'] + ' (' + metrics_global['selected_candidate'] + ')'
    metrics_by_dataset['selected_candidate_label'] = metrics_by_dataset['preprocessing_strategy'] + ' + ' + metrics_by_dataset['model'] + ' (' + metrics_by_dataset['selected_candidate'] + ')'

    if not fold_df.empty:
        fold_df['preprocessing_strategy'] = row['strategy']
        fold_df['selected_candidate_label'] = fold_df['preprocessing_strategy'] + ' + ' + fold_df['model'] + ' (' + fold_df['selected_candidate'] + ')'
        fold_frames.append(fold_df)

    global_frames.append(metrics_global)
    dataset_frames.append(metrics_by_dataset)

all_global = pd.concat(global_frames, ignore_index=True) if global_frames else pd.DataFrame()
all_by_dataset = pd.concat(dataset_frames, ignore_index=True) if dataset_frames else pd.DataFrame()
all_folds = pd.concat(fold_frames, ignore_index=True) if fold_frames else pd.DataFrame()

print(f"Loaded {len(all_global)} aggregated rows from {len(global_frames)} k-fold-by-client strategy runs.")
print(f"Loaded {len(all_by_dataset)} aggregated dataset rows from {len(dataset_frames)} k-fold-by-client strategy runs.")

## Final Comparison Table


In [ ]:
display_columns = ['accuracy','balanced_accuracy','specificity','precision','recall','f1','roc_auc','pr_auc','far','miss_rate']
if all_global.empty:
    final_comparison = pd.DataFrame(columns=['selected_candidate'])
else:
    final_comparison = (
        all_global[['selected_candidate_label','preprocessing_strategy','model','model_candidate','accuracy','balanced_accuracy','specificity','precision','recall','f1','roc_auc','pr_auc','far','miss_rate','tn','fp','fn','tp']]
        .rename(columns={'selected_candidate_label': 'selected_candidate'})
        .sort_values(ranking_columns, ascending=ranking_ascending)
        .reset_index(drop=True)
    )
display(final_comparison if final_comparison.empty else final_comparison.assign(**{c: final_comparison[c].round(4) for c in display_columns}))

## Results by Dataset


In [ ]:
if all_by_dataset.empty:
    dataset_comparison = pd.DataFrame(columns=['dataset', 'selected_candidate'])
else:
    dataset_comparison = (
        all_by_dataset[['dataset','selected_candidate_label','preprocessing_strategy','model','model_candidate','accuracy','balanced_accuracy','specificity','precision','recall','f1','roc_auc','pr_auc','far','miss_rate']]
        .rename(columns={'selected_candidate_label': 'selected_candidate'})
        .sort_values(['dataset', *ranking_columns], ascending=[True, *ranking_ascending])
        .reset_index(drop=True)
    )
display(dataset_comparison if dataset_comparison.empty else dataset_comparison.assign(**{c: dataset_comparison[c].round(4) for c in display_columns}))

## Results by Fold


In [ ]:
if all_folds.empty:
    display(Markdown('Fold-level files are not available yet.'))
else:
    fold_view = (
        all_folds[['fold','selected_candidate_label','preprocessing_strategy','model','accuracy','balanced_accuracy','f1','roc_auc','pr_auc','far','miss_rate']]
        .rename(columns={'selected_candidate_label': 'selected_candidate'})
        .sort_values(['fold', *ranking_columns], ascending=[True, *ranking_ascending])
        .reset_index(drop=True)
    )
    display(fold_view.assign(**{c: fold_view[c].round(4) for c in display_columns if c in fold_view.columns}))